# Fonksiyonlar - Aşırı Kapsamlı

### Motivasyon: 
- Bölüm 3'ün sonlarına doğru birçok kavram birbirine karıştı. Tıkandım...
- Closure, decorators, higher order functions vs. birçok python kavramını yeteri kadar iyi anlayamadım. Bu halde bölüm 4'e geçmek istemiyorum.
- Şimdi Python'u kenara bırakıp sadece fonksiyonlarına çalışacağım. En temelden başlayarak!!!
- Fonksiyonların altındaki mantığı anlayarak sadece Python özelinde değil, genel olarak matematik/diğer yazılım dilleri/kütüphaneler özelinde de soyutlama becerimi geliştireceğimi düşünüyorum.

--- 
### Konu Haritası:

```markdown
1. Önkoşullar - Fonksiyonların Birinci Sınıf Vatandaş Olması 
    1. Fonksiyonların Nesne Olarak İncelenmesi
    2. Scope ve Scope Zinciri
    3. Esnek Argüman Yapıları
2. Higher-Order Functions
    1. Yerleşik HOF'lar
    2. Kendi HOF'larımızı Tasarlamak
3. Closures (Kapanışlar)
    1. Closure Temelleri
    2. Closure Tuzakları ve Gelişmiş Kullanımları
4. Decorators
    1. Decorator Temelleri
    2. Parametreli ve Çoklu Decoratorler
5. İlişkili İleri Konular
    1. functools Ekosistemi
    2. Decorator'e Komşu Kalıplar  
```

--- 

## Bölüm 1.1: Fonksiyonların Nesne Olarak İncelenmesi
---
### *Fonksiyon referansları ve değişkenlere atama*
* Python'da fonksiyonlar birinci sınıf vatandaştır. Yani bir fonksiyon adı, aslında bellekteki bir fonksiyon nesnesine bağlı bir referanstır.
    * def ile bir fonksiyon tanımladığımızda python bu nesneyi oluşturur ve adı ona bağlar; bu adı başka bir değişkene atadığımızda bellekteki aynı nesneyi gösteren ikinci bir referans edersin, "fonksiyonu kopyalamazsın!".
* `isim` fonksiyon nesnesinin kendisini işaret eder, `isim()` ise o fonksiyonu çağırır (call).    

In [18]:
def selamla(ad):
    return f"Merhaba, {ad}!"

# 'selamla' adı, fonksiyon nesnesine bir referanstır.
mesaj_uret = selamla    # kopyalama değil, fonksiyon nesnesine ikinci bir isim/referans oluşturma

print(selamla)                  # Beklenen çıktı: <function selamla at 0x...>
print(mesaj_uret is selamla)    # Beklenen çıktı: True
# 'is' operatoru 2 ismin aynı nesneyi işaret ettiğini kanıtlıyor.
print(mesaj_uret("Ayşe"))       # Beklenen çıktı: Merhaba, Ayşe!

<function selamla at 0x00000182DAE3B420>
True
Merhaba, Ayşe!


### *Fonksiyonu parametre olarak geçirme*

* Bir fonksiyonu başka bir fonksiyona argüman olarak verebilmek, "davraışı parametreleştirme" fikrinin temelidir. Yani ne yapacağını değil, nasıl yapacağını dışarıdan enjekte edebiliriz.
* Python'da bir fonksiyon parametresi, tıpkı bir int ya da str parametresi gibi herhangi bir birinci sınıf vatandaşı kabul edebilir ki fonksiyonlar da birinci sınıf vatandaştır.
    * Bir fonksiyonu parametre olarak alan fonksiyona <u>higher-order function</u> denir.   

In [19]:
def kare_al(x):
    return x*x

def kup_al(x):
    return x*x*x

def uygula(fonksiyon, deger):    # Higher Order Function   
    # burada fonksiyon: herhangi bir tek-argumanlı çağrılabilir nesne olabilir.
    return fonksiyon(deger)

print(uygula(kare_al, 5))
print(uygula(kup_al, 5))

# Aynı uygula fonksiyonu hiç içine müdahale etmeden farklı davranışlar üretebiliyor.
# Çünkü davranışın kendisi (kare_al ya da kup_al) çağrı anında parametre olarak akıyor.  


25
125


### *Fonksiyonu return değeri olarak döndürme*

* Bir fonksiyon tıpkı bir sayı ya da liste döndürebildiği gibi, başka bir fonksiyonu da return ile verebilir. 
* Bir fonksiyonun gövdesi içinde başka bir fonksiyon tanımlayıp (iç içe fonksiyon / nested fonksiyon) onu return edebiliriz.
    * İç fonksiyon, dış fonksiyonun çalıştığı anda her çağrıda yeniden oluşturulur - yani dış fonksiyonu iki kez çağırırsak iki farklı iç fonksiyon nesnesi elde ederiz. Bu davranış, bir fonksiyonun önceden yapılandırılmış özel bir versiyonunu üretmek için kullanılır.
        * Buna genellikle fonksiyon fabrikası (function factory) denir.  

In [20]:
def carpan_uretici(katsayi):    # outer function
    def carpan(sayi):
        return sayi * katsayi
    return carpan   # Fonksiyonun kendisini döndürüyor. Parantez yok - çağrılmıyor!

ikiyle_carp = carpan_uretici(2)     
# Fonksiyon nesnesi döndürür ve döndürülen nesnenin referansı ikiyle carp'a atanır.
# Eğer 'return carpan()' yazsaydık fonksiyon nesnesi değil değer döndürürdü ve o değeri atardık. 
ikiyle_carp_alternatif = carpan_uretici(2)      # Her çağrı başka nesne üretir.
ucle_carp = carpan_uretici(3)

print(ikiyle_carp(5))
print(ucle_carp(5))

print(ikiyle_carp is ikiyle_carp_alternatif)    # Çıktı: False
print(ikiyle_carp is ucle_carp)                 # Çıktı: False

10
15
False
False


## Bölüm 1.2: Scope ve Scope Zinciri
---
### *LEGB Kuralı*

L: Local -> İçinde bulunulan fonksiyonun kendi yerel kapsamı <br>
E: Enclosing -> Bu fonksiyonu saran -eğer varsa- dış fonksiyonun kapsamı<br>
G: Global -> Modül seviyesi <br>
B: Built-in -> Python'un kendi yerleşik isimleri: len, print gibi <p>
* Python isim çözümlemesi (name resolution) yaparken dört katmanı sırasıyla tarar: <br> 
Local -> Enclosing -> Global -> Built-in
* Python ilk eşleşen katmanda durur ve daha dış bir katmana devam etmez. Bu sıralama statikdir, kodun nerede yazıldığına bakılarak belirlenir. Çalışma zamanında hangi fonksiyonun çağrıldığına göre belirlenmez.

In [21]:
x = "global x"

def outer_function():
    x = "enclosing x"
    def inner_function():
        x = "local x"
        print(x)        

    inner_function()    # Local katmanda bulundu, aramaya devam etmez.
    print(x)            # outer_function'un kendi yerel (inner'in enclosing) x'i

outer_function()
print(x)                # Bu da global katmanda bulur.

def outer_function_v2():
    y = "enclosing y"
    def inner_function_v2():
        print(y)        # local katmanda bulamadı, bir üstte enclosing'e bakar.

    inner_function_v2() 
    # outer_function_v2 icin print(y)'yi, y'nin enclosing y value'si ile paketleyerek döndürür. 

outer_function_v2()    # Beklenen çıktı: enclosing y

local x
enclosing x
global x
enclosing y


### *global ve nonlocal anahtar kelimeleri*
* LEGB kuralı isim okumak için geçerlidir; ama bir dış katmandaki değişkene yazmak (ona atama yapmak) istediğimizde Python'a bunu açıkça söylememiz gerekir - iste global ve nonlocal bu ihtiyaç için var.
* Varsayılan olarak bir fonksiyon içinde bir değişkene atama yaptığımızda Python bu değişkeni otomatik olarak local kabul eder - dış kapsamda aynı isimde bir değişken olsa bile.
    * global anahtar kelimesi, bir ismin global (modül) katmanına ait olduğunu oraya yazılacağını bildirir; nonlocal ise bir ismin en yakın enclosing katmana ait olduğunu ve oraya yazılacağını bildirir.
    * İkisi de yalnızca atama için gereklidir. Sadece okumak için hiçbir anahtar kelimeye ihtiyaç yoktur, LEGB zaten okumayı otomatik olarak çözer.
    <p>
* global / nonlocal bildirimini unutup dış değişkene atama yapmaya çalışmak, okuma zamanı değil derleme zamanı bir hataya yol açar — Python fonksiyonu derlerken sayac 'ı zaten yerel olarak işaretler, bu yüzden hata += satırında değil, sayac 'a ilk erişildiği anda ortaya çıkar:

In [22]:
sayac = 0

def global_arttir_hatali():
    sayac += 1          
    # Python sayac'i local kabul eder. global değil local sayaci arttirir. global sayac'a dokunmaz.
                          
# global_arttir_hatali();
# UnboundLocalError verdi. local variable olan 'sayac'a ulaşamıyorum local kapsamda sayac bir value
# ... ile ilişlilendirilmemiş diyor. 

def global_arttir():
    global sayac
    sayac += 1

global_arttir(); global_arttir(); global_arttir()
print(sayac)    # Çıktı: 3


def outer_function():
    sayac = 0

    def inner_function():
        nonlocal sayac
        sayac += 1

    inner_function()
    inner_function()

    return sayac

print(outer_function())     # Çıktı: 2

3
2


## Bölüm 1.3: Esnek Argüman Yapıları
---
### *lambda ifadeleri*
* lambda, tek satırlık, isimsiz bir fonksiyon tanımlamanın kısa yoludur. Özellikle bir fonksiyonu yalnızca bir kez, başka bir fonksiyona argüman olarak geçirmek için kullanacaksak def yazmak gereksiz hale gelir.

`lambda parametreler: ifade` sözdizimi, bellekte def ile tanımlanan sıradan bir fonksiyonla aynı fonksiyon nesnesini üretir. Aralarındaki tek fark isimlendirme ve gövde kısıtıdır.
* Bir lambda gövdesinde yalnızca tek bir ifade olabilir ve bu ifadenin sonucu otomatik olarak return edilir. Döngü, control yapısı, birden fazla satır içermez.
* Koşullu ifade expressionu olarak `x if condition else y` şeklinde kullanılabilir.


In [23]:
kare_lambda = lambda x:x**2             # İsme referansını atamak lambda için kötü pratik
print(kare_lambda(6))                   # Çıktı: 36

def kare_def(x):
    return x**2

print(kare_lambda(4)==kare_def(4))      # Çıktı: True

# lambda ifadelerinin asıl işlevi fonksiyon nesnesi olarak argüman verilmesi -anlık kullanım-
kelimeler = ["muz", "elma", "kivi", "armut"]
print(sorted(kelimeler, key=lambda k:len(k)))  # sorted(dizi, fonksiyon), dizi döndürür.

36
True
['muz', 'elma', 'kivi', 'armut']


### _*args ve **kwargs_

* Bir fonksiyonun kaç argüman alacağını önceden bilmediğimiz durumlarda (örn. bir yüksek dereceli fonksiyona hangi argümanların geleceğini bilmiyorsak) *args ve **kwargs fonksiyona değişken sayıda argüman kabul etme yeteneği kazandırır. 
* Packing:
    - *args -> Fazladan gelen tüm positional argümanları "args" isimli bir tuple içinde toplar.
    - **kwargs -> Fazladan gelen tüm keyword argümanları "kwargs" isimli bir dict içinde toplar. 
* Unpacking:
    - (*) ve (**) operatörleri çağrı tarafında da kullanılabilir. Bu sefer var olan bir tuple/dict'i argümanlara açar.  

In [24]:
# Packing
def bilgi_yazdir(*args, **kwargs):
    print("Positional:", args)
    print("Keyword:", kwargs)

bilgi_yazdir(1, 2, "üç", ad="Ayşe", yas=30)

# Unpacking
def toplam(a, b, c):
    print(a+b+c)

sayilar = (1, 2, 3)
toplam(*sayilar)
parametreler = dict(a=10, b=20, c=30)
toplam(**parametreler)

Positional: (1, 2, 'üç')
Keyword: {'ad': 'Ayşe', 'yas': 30}
6
60


## Bölüm 2.1: Yerleşik HOF'lar
---
### *map()*

* map(), bir koleksiyondaki her elemana aynı dönüşümü uygulamanın döngü yazmadan standart yoludur (Vektörizasyon işe MATLAB'deki). Bir fonksiyonu, bir iterable'ın her elemanına "haritalar".
* map(fonksiyon, iterable) çağrısı, fonksiyon'u iterable'ın her elemanına sırasıyla uygular ve sonuçları içeren bir map nesnesi döndürür. 
    * Bu nesne lazy evaluate edilir yani elemanlar map() çağrıldığı anda değil, üzerinde gezinildiğinde (bir döngüde, list() içine konulduğunda) tek tek hesaplanır. Bu, map'i belleği verimli kullanan bir iterator yapar; büyük veri setlerinde tüm sonucu aynı anda belleğe yüklemeden işlem yapmayı sağlar.
    * map nesnesi bir kere tüketilir (one-shot iterator) - üzerinde bir kez list() çağırdıktan sonra tekrar list() çağırırsak boş liste alırız, çünkü iterator zaten bitmiştir.  <p>
* map() birden fazla itarable da kabul edebilir - bu durumda fonksiyon, her iterable'dan aynı index'teki elemanları paralel olarak alır. Aralarından kısa olan iterable bitince durur.
> Çoğu dönüşüm geliştiricilerce daha okunaklı bulunan list comprehension ile yazılabilir ama map lazy iken list comprehension anında bir liste üretir. Büyük veri setleri ile çalışırken bu düşünülmeli 

In [25]:
sayilar = [1, 2, 3, 4, 5]

karesi_alinmis = map(lambda x:x**2, sayilar)
print(type(karesi_alinmis))                         # Çıktı: <class 'map'>
print(karesi_alinmis)                               # Beklenen çıktı: <map object at 0x...>
print(list(karesi_alinmis))                         # Çıktı: [1, 4, 9, 16, 25]
print(list(karesi_alinmis))                         # Çıktı: []

a = [1, 2, 3]
b = [10, 20, 30]
toplamlar = map(lambda x,y:x+y, a, b)
print(list(toplamlar))

<class 'map'>
[1, 4, 9, 16, 25]
[]
[11, 22, 33]


### *filter()*

* map() her elemanı dönüştürürken, filter elemanı belli bir koşula göre eleyip geçirir (Bu da MATLAB'deki maskeleme oluyor). Koleksiyondan yalnızca belirli bir şartı sağlayan elemanları seçmenin standart yoludur.
* filter(fonksiyon, iterable) çağrısı fonksiyon'u iterable'ın her elemanına uygular ve bu fonksiyon True (ya da Truthy bir değer) döndüren elemanları içeren bir "filter nesnesi" döndürür. Aynı tüketim kuralları burada da geçerlidir. 
    * Buradaki fonksiyon her zaman tek bir argüman alıp boolean değer döndürmelidir; bu evet/hayır kararı veren fonksiyonlara predicate(yüklem) denir. <p>
* Argüman olarak fonksiyon yerine None koyabiliriz: bunu yaptığımızda bütün false ve falsy değerleri listeden temizler. Kullanışlı bir yöntemdir. 


In [26]:
sayilar = range(1,11)
ciftler = filter(lambda x:x%2 == 0, sayilar)
print(list(ciftler))     # Beklenen çıktı: [2, 4, 6, 8, 10]

# Boş değerleri listeden temizlemek
karisik = [0, 1, "", "merhaba", None, [], [1,2], False, True]
temizlenmis = filter(None, karisik)
print(temizlenmis)          # <filter object at 0x0000020C02144C10>
print(list(temizlenmis))    # [1, 'merhaba', [1, 2], True]
print(list(temizlenmis))    # []

# map ve filter birlikte zincirleme kullanım -> önce ele sonra dönüştür
sayilar = range(1,11)
sonuc = list(map(lambda x:x**2, filter(lambda x:x%2==0, sayilar)))
print(sonuc)    # Aralarından çift olanları seçer ve karelerini alır.

[2, 4, 6, 8, 10]
[1, 'merhaba', [1, 2], True]
[]
[4, 16, 36, 64, 100]


### *sorted(), min(), max() fonksiyonlarının key parametresi ile kullanımı*

* sorted(), min() ve max() fonksiyonları, elemanları doğrudan sıralamak yerine, her elemandan sıralanacak değerlerini üreten fonksiyonu (kendileri "key" olur) parametre olarak kabul eder. 
* Bu, bir Higher Order Function'un en yaygın kullanım biçimlerinden biridir, sıralama mantığını dışarıdan enjekte edebiliriz. **sorted() fonksiyonunun kodlarına içeriden müdahale etmeden hem de!!!** <p>
* key parametresine geçiriken her fonksiyon, her elemanı alır ve karşılaştırma için kullanılacak bir değer döndürür. sorted/min/max elemanların kendisini değil, bu fonksiyonun her eleman için ürettiği sonuçları karşılaştırır.
    * Varsayılan olarak -key fonksiyonu tanımlanmadığında- Python bu elemanları < operatörüyle küçükten büyüğe doğru doğrudan karşılaştırır.
    * Karşılaştırılmayan tipler (örn. dict'ler) için key kullanmak zorunludur, yoksa TypeError fırlatır.
    * str, int gibi birbiriyle karşılaştırılamayan tiplerin aynı anda bulunduğu list gibi veri yapılarında da sorted() aynı TypeError hatasını fırlatır. <p>

> sorted() **fonksiyonu** orijinali bozmadan yeni liste döndürürken, sort() in-place **metottur**, orijinal listeyi değiştirir ve None döndürür. 



In [27]:
sayilar = [13, 4, 35, 7, 3, 28, 5]
kelimeler = ["muz", "elma", "kivi", "armut", "vişne"]

print(sorted(sayilar))          # default: küçükten büyüğe  
print(sorted(kelimeler))        # default: ascii küçükten büyüğe
print(sorted(sayilar, key=lambda x:-x)) # key ile büyükten küçüğe
print(sorted(sayilar, key=lambda x: x%5))   # 5 ile kalanına göre
print(sorted(kelimeler, key=len))     # kelimelerin uzunluğuna göre

# min/max da aynı key mantığını paylaşır. Dışarıdan sıralama mantığı verebiliriz.
print(min(kelimeler, key=len))  # uzunluklarına göre sıralandığında min
print(max(kelimeler, key=len))  # ... max

ogrenciler = [
    {"ad":"Sude", "not":78},
    {"ad":"Yusuf", "not":92},
    {"ad":"Esmanur", "not":85},
    {"ad":"Samet", "not":68},
    {"ad":"Emir", "not":74},
]

print(sorted(ogrenciler, key=lambda ogr: ogr["not"], reverse=True))
# öğrencilerin olduğu sözlükleri en yüksek not alandan en düşük not alana doğru sıralar

sayilar = [5, 2, 8, 1]
yeni = sorted(sayilar, key=lambda x:-x)
print(sayilar, yeni)    # 'sayilar' listesi değişmedi.

donus = sayilar.sort(key=lambda x:-x)
print(sayilar, donus)   # 'sayilar' listesi değişti bu sefer. None döndürdü.

[3, 4, 5, 7, 13, 28, 35]
['armut', 'elma', 'kivi', 'muz', 'vişne']
[35, 28, 13, 7, 5, 4, 3]
[35, 5, 7, 13, 3, 28, 4]
['muz', 'elma', 'kivi', 'armut', 'vişne']
muz
armut
[{'ad': 'Yusuf', 'not': 92}, {'ad': 'Esmanur', 'not': 85}, {'ad': 'Sude', 'not': 78}, {'ad': 'Emir', 'not': 74}, {'ad': 'Samet', 'not': 68}]
[5, 2, 8, 1] [8, 5, 2, 1]
[8, 5, 2, 1] None


### *functools.reduce()*
* map() ve filter() HOF'ları birer iterator olup lazy evaluation yaparken, reduce() HOF'u lazy değildir direkt o an sonucu bize verir. Ne de olsa elemanlar üretmez/filtrelemez, tüm koleksiyonu tek bir elemana indirger.
* Yine diğer iki HOF gibi built-in değildir. functools modülünden import edilmesi gerekiyor. 
- reduce(fonksiyon, iterable, baslangıc=...) çağrısı fonksiyonu iki argümanla (bir biriktirici ve sıradaki eleman) sırayla çağırır. Önce ilk iki elemanla başlar sonucu bir sonraki elemanla tekrar fonksiyona geçirir ve bu böyle tüm koleksiyon bitene kadar devam eder.
    - baslangic parametresi opsiyoneldir ama iki önemli rolü vardır.
        1. boş bir itarable için reduce'un hata vermeden bir sonuç döndürmesini sağlar.
        2. biriktiricinin (accumulator) başlangıç tipini/değerini netleştirir. 

> reduce, en genel HOF'tur. aslında sum(), max() hatta belirli bir açıdan map/filter bile kavramsal olarak reduce'un özelleştirilmiş hali sayılabilir. Ama python topluluğu sum(), bir döngü ya da comprehension'u dahi reduce'a tercih eder çünkü reduce ile yazılan kod çoğu zaman daha az okunaklıdır.

In [28]:
from functools import reduce 

sayilar = [1, 2, 3, 4, 5]

toplam = reduce(lambda acc,x:(acc+x), sayilar)
print(toplam)       # Çıktı: 15

toplam = reduce(lambda acc,x:(acc+x), sayilar, 50)
print(toplam)       # Çıktı: 65

# başlangıç parametresi olmadığında hata alma ihtimalimiz var.
try:
    print(reduce(lambda acc,x: acc+x, []))
except TypeError as hata_mesaji:
    print(f"Hata: {hata_mesaji}")

# başlangıç değeri ile boş liste - güvenli
print(reduce(lambda acc,x: acc+x, [], 0))    

15
65
Hata: reduce() of empty iterable with no initial value
0


## Bölüm 2.2: Kendi HOF'larımızı Tasarlamak
---
### *Fonksiyon fabrikaları (fonksiyon döndüren fonksiyon)*

Bölüm 1'de return ile fonksiyon nesnesi döndürebilen fonksiyonlar görmüştük. 
* Fonksiyon fabrikası, fonksiyondan fonksiyon döndürme mekanizmasının parametreye göre özelleşmiş fonksiyonlar üretmek amacıyla tasarım kalıbına dönüştürülmüş halidir.
* Bir fonksiyon fabrikası dışarıdan bir veya birkaç yapılandırma parametresi alır ve bu parametrelere göre önceden ayarlanmış davranan yeni bir fonksiyon üretip döndürür. 
    * Mekanizma şu şekilde:
    - İç fonksiyon dış fonksiyonun parametrelerine ve varsa local değişkenlerine erişimini korur - dış fonksiyon çoktan return ile bitmiş ve stack'ten silinmiş olsa bile...
    - Normalde dış fonksiyonların değişkenleri de fonksiyon bittiğinde kendisiyle beraber silinmesi beklenir ama iç fonksiyon hala o değişkenlere ihtiyaç duyduğu için Python bunları iç fonksiyonla birlikte canlı tutar -> Bu davranışın adı "Closure"
    - Bölüm 3'te mekanizmasını `__closure__` özniteliğiyle daha detaylı inceleyeceğiz. Şimdilik amaç kalıbın nasıl sakladığını değil, bu saklamanın ne işe yaradığını görmek.


In [29]:
def indirim_hesaplayici_uret(oran):     # FONKSİYON FABRİKASI 
    """oran'a göre özelleştirilmiş bir indirim hesaplayıcı üretir."""
    def hesapla(fiyat):
        return fiyat*(1-oran)
    return hesapla

yuzde_10_indirim = indirim_hesaplayici_uret(0.10)
yuzde_25_indirim = indirim_hesaplayici_uret(0.25)   
# indirim_hesaplayıcı_uret özelleştirilmiş hesapla'yı döndürüyor scope'u bitiyor. 
# indirim_hesaplayıcı_uret'in "oran" değişkeni hesapla için lazım hala...
# closure mekanizması bu "oran" değişkeninin silinmesini önlüyor.

print(yuzde_10_indirim(200))
print(yuzde_25_indirim(200)) 

# Fonksiyon fabrikalarının asıl gücü -> birden fazla parametreyle fonksiyonları özelleştirmek
def dogrulayici_uret(min_deger, max_deger):
    def dogrula(deger):
        kontrol = (min_deger <= deger <= max_deger)
        return kontrol
    return dogrula

yas_dogrula = dogrulayici_uret(0 ,120)
oran_dogrula = dogrulayici_uret(0, 100)

print(yas_dogrula(45))
print(oran_dogrula(-15))

180.0
150.0
True
False


## Bölüm 3.1: Closure Temelleri
---
### *Free variable ve closure mekanizması, \_\_closure\_\_ incelemesi*

* Closure: <br>
  Bir iç fonksiyonun, kendisini saran fonksiyon çoktan bitmiş ve stackten silinmiş olsa bile, o fonksiyonun kendisine sağladığı değişkenlere erişimini koruyabilmesi olayıdır.

* Serbest Değişken (Free Variable):    
  Bir iç fonksiyonun gövdesinde kullanılan ama ne kendi local kapsamında ne de global kapsamda tanımlı olan, dış fonksiyonun enclosing kapsamından gelen değişkenlere denir. <p>

- Python, bir fonksiyon bu tür serbest değişkenler içeriyorsa onu sıradan bir fonksiyon olarak değil, bir "closure" olarak oluşturur.
- Fonksiyon nesnesi kullandığı free variable'ların değerlerine referans veren (hatırlatma) küçük kapsayıcılar -cell nesneleri- taşır. Bu bilgi de oluşturulan fonksiyonun \_\_closure\_\_ özniteliğinde (Java OOP'deki field işte) saklanır.

> Hatırlatma: Python'da değişkenler C'deki gibi sadece kendilerine ait olan değerlerini tutmazlar. O değeri temsil eden ve tek olan nesnenin hafızadaki konumunu gösteren referanslardır. Call stack bittiği zaman fonksiyonların o referansları/etiketleri silinir ama gösterdikleri asıl nesne silinmez.  <p>


- Hangi değişkenlerin saklandığını da \_\_code\_\_.co_freevars ile görebiliyoruz, direkt deişkenlerin identifierlarını, isimlerini veriyor.

In [30]:
def hesap_olustur(hesap_sahibi, baslangic_bakiyesi):
    islem_sayisi = 0                            # 3 tane free variable'ımız var.

    def islem_yap(miktar):
        nonlocal baslangic_bakiyesi, islem_sayisi 
        # hesap_sahibi'ni sadece okuyacağımız için nonlocal dememize gerek yok, LEGB çalışır.
        baslangic_bakiyesi += miktar
        islem_sayisi += 1
        return f"{hesap_sahibi} | Bakiye: {baslangic_bakiyesi} TL | İslem Adedi: {islem_sayisi}"
    return islem_yap

# Bağımsız Hesaplar -Fonksiyon Nesneleri- Oluşturuyoruz.
# Her biri hafızada yepyeni hücreler barındıran farklı closure'lar üretir.
ali_hesap = hesap_olustur("Ali", 1000)
ayse_hesap = hesap_olustur("Ayşe", 5000)

# Python free variable'ları alfabetik sıraya göre dizer.
print("Ali'nin Değişkenleri", ali_hesap.__code__.co_freevars)
print("Ali İndeks 0 (Bakiye):", ali_hesap.__closure__[0].cell_contents)
print("Ali İndeks 1 (İsim):", ali_hesap.__closure__[1].cell_contents)
print("Ali İndeks 2 (İşlem):", ali_hesap.__closure__[2].cell_contents)

print("Ayşe'nin Değişkenleri", ayse_hesap.__code__.co_freevars)
print("Ayşe İndeks 0 (Bakiye):", ayse_hesap.__closure__[0].cell_contents)
print("Ayşe İndeks 1 (İsim):", ayse_hesap.__closure__[1].cell_contents)
print("Ayşe İndeks 2 (İşlem):", ayse_hesap.__closure__[2].cell_contents)

# State değişimi - nonlocal sayesinde
print("İşlem 1:", ali_hesap(200))
print("İşlem 2:", ali_hesap(-50))
# Değişim Sonrası Cell Kontrolü
print("Ali Yeni İndeks 0 (Bakiye):", ali_hesap.__closure__[0].cell_contents, "TL")
print("Ali Yeni İndeks 2 (İşlem):", ali_hesap.__closure__[2].cell_contents, "adet")
# Bağımsızlık: Ali'nin İşlemleri Ayşenin hücrelerini asla etkilemez.
# İkisinin de bakiye, islem hücreleri üretilen closure'a özgü farklı/bağımsız hücrelerdir. 
print("Ayşe İndeks 0 (Bakiye):", ayse_hesap.__closure__[0].cell_contents, "TL (Hiç bozulmadı!)")


# İzole Bellek Alanları 
# ali_hesap ve ayse_hesap tamamen ayrı __closure__ tuple'larına sahiptir. 
# Biri değiştiğinde diğerinin hücre nesneleri bundan habersizdir demiştik zaten üstte de
print("--- İzole Bellek Alanları ---")
print(ali_hesap.__closure__)    # çıktısı tuple'dır. İçinde cell referansları vardır.
print(ayse_hesap.__closure__)


Ali'nin Değişkenleri ('baslangic_bakiyesi', 'hesap_sahibi', 'islem_sayisi')
Ali İndeks 0 (Bakiye): 1000
Ali İndeks 1 (İsim): Ali
Ali İndeks 2 (İşlem): 0
Ayşe'nin Değişkenleri ('baslangic_bakiyesi', 'hesap_sahibi', 'islem_sayisi')
Ayşe İndeks 0 (Bakiye): 5000
Ayşe İndeks 1 (İsim): Ayşe
Ayşe İndeks 2 (İşlem): 0
İşlem 1: Ali | Bakiye: 1200 TL | İslem Adedi: 1
İşlem 2: Ali | Bakiye: 1150 TL | İslem Adedi: 2
Ali Yeni İndeks 0 (Bakiye): 1150 TL
Ali Yeni İndeks 2 (İşlem): 2 adet
Ayşe İndeks 0 (Bakiye): 5000 TL (Hiç bozulmadı!)
--- İzole Bellek Alanları ---
(<cell at 0x00000182DA7BC0A0: int object at 0x00000182DAE3DCB0>, <cell at 0x00000182DA7BE2F0: str object at 0x00000182DA5AC870>, <cell at 0x00000182DA7BC6D0: int object at 0x00007FFA8B68A3C8>)
(<cell at 0x00000182DA7BEE00: int object at 0x00000182DAE3D190>, <cell at 0x00000182DA7BFE20: str object at 0x00000182DAEF17A0>, <cell at 0x00000182DA7BC580: int object at 0x00007FFA8B68A388>)


### *nonlocal ile closure state'i değiştirme*

* nonlocal, bir closure'ın free variable'larının cell içeriğini kalıcı olarak deiştirmesini sağlar. Yani closure, salt bir anlık okuma değil, çağrılar arasında hafızası olan bir durum (state) taşıyabilir.

> Enclosing kapsamdaki bir değişkene, nonlocal declare etmeden atama yapmaya çalışırsak local kapsamda yeni bir değişken oluştururuz. Bu da UnboundLocalError hatasına sebep olur.

* nonlocal declarasyonu, Closure'ları sayaç, biriktirici veya basit önbellek gibi durumlu(stateful) araçlar yapmanın altın yoludur.
* Normalde bu amaçla sınıf da yazılabiliyor ama closure tek bir değişken tutmamızın gerektiği durumlar için daha hafif bir alternatiftir.
<p>

* Bu kalıp, birden fazla iç fonksiyonun aynı cell'i paylaşmasına da izin verir.
    * Enclosing kapsamındaki bir değişkeni birden fazla iç fonksiyon kullanırsa hepsi aynı cell üzerinden okuma/yazma yapar.   

> Aşağıdaki banka_hesabi_uret örneği closure'ların bir encapsulation(OOP) aracı olarak da kullanılabileceğini gösterir: bakiye, dışarıdan doğrudan erişilemez - yalnızda yatir/cek üzerinden erişilebilir.
> * Tradeoff'u closure'ların yalnızca encapsulation'a olaanak tanıyıp inheritance veya polymorphism gibi bir sınıfın sunduğu genişletilebilirlik yeteneklerine sahip olmamasıdır.
> * Küçük, sabit sayıda davranış için closure yeterlidir. 

In [31]:
def sayac_uret():
    sayi = 0
    def arttir():
        nonlocal sayi
        sayi += 1
        return sayi
    return arttir

sayac1 = sayac_uret()
print(sayac1())
print(sayac1())
print(sayac1())

sayac2 = sayac_uret()   # Farklı Closure'ların cell'leri farklı
print(sayac2())         # Çıktı: 1
print(sayac1())         # Çıktı: 4

print("Sayaç1 Cell İçerik:", sayac1.__closure__[0].cell_contents)
sayac1()
print("Sayaç1 Cell İçerik:", sayac1.__closure__[0].cell_contents)

# Birden fazla iç fonksiyonun aynı değişken üzerinden ortak okuma/yazma yapması
def banka_hesabi_uret(baslangic_bakiye):
    bakiye = baslangic_bakiye

    def yatir(miktar):
        nonlocal bakiye
        bakiye += miktar
        return bakiye
     
    def cek(miktar):
        nonlocal bakiye
        if miktar>bakiye: raise ValueError("Yetersiz Bakiye")
        bakiye -= miktar
        return bakiye

    return yatir, cek   # Bu iki fonksiyon aynı 'bakiye' cell'ini paylaşıyor.

yatir, cek = banka_hesabi_uret(100)
print(yatir.__closure__[0] is cek.__closure__[0])   #  Çıktı: True, İkisi de Bellekteki aynı nesnedir.

print(yatir(50))
print(cek(30))

try: 
    cek(1000)
except ValueError as hata_mesaji:
    print(f"Hata: {hata_mesaji}")


1
2
3
1
4
Sayaç1 Cell İçerik: 4
Sayaç1 Cell İçerik: 5
True
150
120
Hata: Yetersiz Bakiye


## Bölüm 3.2: Closure Tuzakları ve İleri Kullanım
---
### *Late binding tuzağı (döngü içinde closure)* 

* Fonksiyonlar tanımlanırlen (ve hemen aynı statementta append edilirken), katsayi'nin değerini okumazlar.
    * Fonksiyon be like: "Bana sadece 'katsayi'nın tutulduğu cell'in referansını ver. Sonrasında çağrıldığında gidip içine bakarım."
    * Fonksiyon her çağrıldığında hepsi gider o aynı ortak hücreye bakar ve oradaki katsayi=3 değerini görüp işlemi yaparlar.
<p>

* Çözüm: Early Binding <br>
    Değeri default argument olarak dondururuz. Bir fonksiyonun default argüman değerleri fonksiyon tanımlandığı anda bir kez hesaplanıp saklanır. Bu farkı kullanarak 'katsayi'yı her iterasyonda yakalayabiliriz.

In [32]:
# --- LATE BINDING TUZAGI ---
def carpanlari_uret():
    carpanlar = []
    for katsayi in [1, 2, 3]:
        def carpan(x):
            return katsayi*x
        carpanlar.append(carpan)
    return carpanlar

uretilen_carpanlar = carpanlari_uret()

print(uretilen_carpanlar[0](10))   # Beklenen 1*10'dan 10 | Çıktı: 30
print(uretilen_carpanlar[1](10))   # Beklenen 2*10'dan 20 | Çıktı: 30
print(uretilen_carpanlar[2](10))   # Beklenen 3*10'dan 30 | Çıktı: 30

print(uretilen_carpanlar[0].__code__.co_freevars)   # Çıktı: ('katsayi', )
print(uretilen_carpanlar[1].__code__.co_freevars)   # Çıktı: ('katsayi', )
print(uretilen_carpanlar[2].__code__.co_freevars)   # Çıktı: ('katsayi', )

print(uretilen_carpanlar[0].__closure__[0] is uretilen_carpanlar[2].__closure__[0]) # Çıktı: True
# Hepsinin 'katsayi' adlı free variable'ı bellekteki aynı nesneymiş.

print(uretilen_carpanlar[0].__closure__[0].cell_contents)   # Hepsinin de cell content'i -> 3

print("-"*20)

# --- Çözüm: EARLY BINDING ---
def carpanlari_uret():
    carpanlar = []
    for katsayi in [1, 2, 3]:
        def carpan(x, katsayi=katsayi):
            return katsayi*x
        carpanlar.append(carpan)
    return carpanlar

uretilen_carpanlar = carpanlari_uret()

print(uretilen_carpanlar[0](10))   # Çıktı: 10
print(uretilen_carpanlar[1](10))   # Çıktı: 20
print(uretilen_carpanlar[2](10))   # Çıktı: 30

print(uretilen_carpanlar[0].__code__.co_freevars)   # Çıktı: ()
print(uretilen_carpanlar[0].__closure__)            # Çıktı: None
# Artık katsayı bir free variable değil, local variable oldu

30
30
30
('katsayi',)
('katsayi',)
('katsayi',)
True
3
--------------------
10
20
30
()
None


### *Closure tabanlı basit memoization örüntüsü*

Closure'un cell içinde state saklayabilmemizi sağlaması şunu da yapabilmemizi sağlıyor -> Memoization 

* Memoization: 
    - Memoization, zaman ve işlem gücü bakımından pahalı olan fonksiyonların daha önce hesaplanmış sonuçlarını bir bellekte (cache) saklama tekniğidir.
    - Fonksiyon aynı argümanlarla tekrar çağrıldığında işlemi baştan yapmak yerine, sonucu doğrudan bu bellekten çekerek performansı önemli ölçüde arttırır.

* Neden Sınıf oluşturmak yerine Closure?
    - Bu önbelleği kodda nerede saklayacağımız hakkında önemli bir mimari karar vardır. Bu sebeple değişkenleri global yapmamalıyız ki dışarıdan değiştirilemesinler. -> Encapsulation
    - Closure'lar da encapsulation yapabilmemizi sağlıyor. Dışarıdan erişilemeyen izole bir bellek alanı -cell- yaratarak!
    - Sırf tek bir sözlük'ü güvene almak için en baştan sınıf oluşturmak için uğraşmamıza, kod kalabalığı yapmamıza gerek yok. Encapsulationu closure ile de yapabiliyoruz, kendi özel state'lerimizi güvenli cell'lerde saklayabiliyoruz.

> Burada dikkat edilmesi gereken ince nokta: dict'in kendisi değiştiriliyor (mutate ediliyor), yeniden atanmıyor — bu yüzden nonlocal bildirimine ihtiyaç yoktur. nonlocal yalnızca atama için gerekir. <p>
> dict[anahtar] = deger bir atama değil, var olan nesne üzerinde bir metod/operasyon çağrısıdır.

* Bir de şunu unutma: cache içeriği hashlable olan argümanlar olmak zorunda. Bir list'i önbelleğe alamayız çünkü liste dict'in key'i olamıyordu mutable veri tipi olduğu için

In [33]:
import time

def memoize_et(fonksiyon):  # Kendisine gönderiken fonksiyona hazırda önbellek tutabilmesi yeteneğini kazandırır.
    cache = {}      # Bu sözlük bizim dışarıdan erişilemeyen güvenli önbelleğimiz

    def wrapper(parametre):
        if parametre in cache:
            return f"[Elimizde ondan hazırda vardı]: Sonuç {cache[parametre]}"
 
        print(f"[{parametre}] için hesaplıyorum bir sn...", end=" ")
        sonuc = fonksiyon(parametre)
        cache[parametre] = sonuc

        return f"[Hesaplama bitti]: Sonuç: {sonuc}"
    return wrapper

def pahali_kare_alma(sayi):
    time.sleep(2)   # Kendimiz gecikme ekliyoruz
    return sayi*sayi

hizli_kare_alma = memoize_et(pahali_kare_alma)  # Fonksiypnu sararak ona memoize yeteneği kazandırıyoruz.

print(hizli_kare_alma(5))       # İlk denemede baştan hesaplar.
print(hizli_kare_alma(5))       # Hazır sonucu hemen verir.
print(hizli_kare_alma(5))       # Hazır sonucu hemen verir.

print(pahali_kare_alma(5))      # Her seferinde en baştan fonksiyonu hesaplar.
print(pahali_kare_alma(5))
print(pahali_kare_alma(5))

print(hizli_kare_alma(9))
print(hizli_kare_alma(4))  
print("Free değişkenler:", hizli_kare_alma.__code__.co_freevars) 
# Alfabetik olarak sıralıyor. cache'in indexi 0 burada, fonksiyon ise 1
print("Bellektekiler:", hizli_kare_alma.__closure__[0].cell_contents)

[5] için hesaplıyorum bir sn... [Hesaplama bitti]: Sonuç: 25
[Elimizde ondan hazırda vardı]: Sonuç 25
[Elimizde ondan hazırda vardı]: Sonuç 25
25
25
25
[9] için hesaplıyorum bir sn... [Hesaplama bitti]: Sonuç: 81
[4] için hesaplıyorum bir sn... [Hesaplama bitti]: Sonuç: 16
Free değişkenler: ('cache', 'fonksiyon')
Bellektekiler: {5: 25, 9: 81, 4: 16}


## Bölüm 4.1: Decorator Temelleri
---
### *Decorator'un closure üzerine inşası ve @ syntax'ı*

* Decorator: <br>
    - Bir önceki kodda elle açık halini yazdığımız wrapper=memoize_et(fonksiyon) kalıbının resmi adıdır.
    - @ sözdizimi yukarıdaki atamayı fonksiyon tanımının hemen üstüne yazarak aynı işi daha okunaklı şekilde yapmamızı sağlayan bir syntatic sugar'dır.
    - Yani decorator'un arkasında ayrı bir mekanizma yoktur. Python'un closure özelliğiyle yapabildiğimiz fonksiyon sarmalamasının syntax'a dökülmüş halidir.

In [34]:
import time
def memoize_et(fonksiyon):
    cache = {}
    def wrapper(parametre):
        if parametre in cache: return f"[Hazırda vardı]: Sonuç {cache[parametre]}" 
        print(f"[{parametre}] için hesaplıyorum bir sn...", end=" ")
        sonuc = fonksiyon(parametre)
        cache[parametre] = sonuc
        return f"[Hesaplama bitti]: Sonuç: {sonuc}"
    return wrapper

@memoize_et
def kare_alma(sayi):
    time.sleep(2)
    return sayi*sayi

# Yukarıdaki şey önceki kodda şu ikisinin kolaylaşırılmış hali:
# 1) kare_alma fonksiyonu tanımı
# 2) kare_alma = memoize_et(kare_alma)      "fonksiyonu aynı ismi kullanarak bir defa bütünüyle sarmalar"
# -> Artık orijinal kare_al işaret edilmez. 

print(kare_alma(3))
print(kare_alma(3))     # Aynı sarmalanmış fonksiyondur.

print(kare_alma.__name__)   # Çıktı: wrapper (kare_alma değil!) -> Bunun olmasını istemeyiz.


# --- Başka bir decorator örneği ---
def logla(fonksiyon):
    def wrapper(*args, **kwargs):       # Farklı Parametreleri olan fonksiyonlarda çalışır.
        print(f"[LOG] {fonksiyon.__name__} çağrılıyor")
        sonuc = fonksiyon(*args, **kwargs)
        print(f"[LOG] {fonksiyon.__name__} tamamlandı → {sonuc}") 
        return sonuc
    return wrapper

@logla                  # decorator ile fonksiyon sarmalama
def topla(a, b):
    return a+b

def cikar(a,b):         # eski şekilde fonksiyon sarmalama
    return a-b
cikar = logla(cikar)

topla(2, 4)
cikar(7, 3)

print(topla.__name__)
print(cikar.__name__)

[3] için hesaplıyorum bir sn... [Hesaplama bitti]: Sonuç: 9
[Hazırda vardı]: Sonuç 9
wrapper
[LOG] topla çağrılıyor
[LOG] topla tamamlandı → 6
[LOG] cikar çağrılıyor
[LOG] cikar tamamlandı → 4
wrapper
wrapper


### *functools.wraps ile metadata korunması*

* Bir üstteki kodda `kare_alma.__name__` dediğimizde bize `wrapper`ismini döndürmesi tesadüfi bir denk geliş değildi! 
    - Decorator ile veya eski usul sarmalanan her fonksiyon metadata'sını kaybeder.
    - Fonksiyon nesnesinin metadata öznitelikleri:
        1. \_\_name\_\_  
        2. \_\_doc\_\_  
        3. \_\_module\_\_  
* functools.wraps orijinal fonksiyon sarıldığında metadatasını kaybetmemesinin standart yoludur.
    - Aslında kendisi bir decorator fabrikasıdır.
    - Wrapper fonksiyonun hemen üstüne @functools.wraps(fonksiyon) yazıyoruz. Burada fonksiyon outer_function'un aldığı orijinal fonksiyondur.
    - Orijinal fonksiyonun metadata özniteliklerini kopyalayıp sarmalayıcıya yapıştırır.
    - Bu sadece kozmetik için uygulanan bir şey değildir. help(kare_alma), kare_alma.\_\_doc\_\_ gibi introspection araçlarının doğru çalışması, debugging ve bazı frameworklerin fonksiyon adına dayanarak düzgün çalışması bu uygulamaya bağlıdır.
    - Her yazdığımız decorator'da functools.wraps kullanmak istisnasız best-practice'dir.

In [35]:
import functools 

def logla(fonksiyon):
    @functools.wraps(fonksiyon)     # metadata'yı fonksiyon'dan wrapper'a kopyalar.
    def wrapper(*args, **kwargs):
        print(f"[LOG] {fonksiyon.__name__} çağrılıyor")
        return fonksiyon(*args, **kwargs)
    return wrapper

@logla 
def kuvvet_alma(a, b):
    """İlk argümanın ikinci argüman'dan olan kuvvetini döndürür."""
    return a**b

print(kuvvet_alma.__name__)
print(kuvvet_alma.__doc__)      # Çıktı olarak docstring'e erişebiliyoruz.
# functools ile wraplemeseydik sonraki aşamalarda debugging yaparken işimize çok yaparacak olan 
# birçok bilgiyi kaybedecektik 


kuvvet_alma
İlk argümanın ikinci argüman'dan olan kuvvetini döndürür.


### *typing.Callable ile fonksiyon tipi belirtme* - Ek Başlık

* Büyüyen bir kod tabanında yazdığımız fonksiyonların hangi tip verileri argüman olarak alıp hangi tip veri döndürmesi niyetiyle yazıldığını; IDE'lere, analiz araçlarına veya kodu okuyan insanlara belirtmek isteyebiliriz.
* Callable, typing modülünden gelir ve tipleri belirtmemizi sağlar.
    - En basitinden Callable tek başına parametre/dönüş tipi belirtmeden kullanılır.
    - Spesifik olarak belirtmek istersek şu şekilde belirtiriz: <br>
    `Callable[[ParametreTipleri], Dönüş Tipi]`   
    - Bu bilgi runtime esnasında Python'a bir şey zorlamaz. Yalnızca IDE'lere, mypy gibi statik tip denetleyicilerine ve okuyan geliştiricilere dokümantasyon sağlar.
* Tip belirteci olmadan dümdüz yazmaya da duck typing deniliyormuş. Küçük projelerde böylesi daha verimliymiş.

In [36]:
# 1) Genel Kullanım - Sadece Callable 

from typing import Callable
import functools

def logla(fonksiyon: Callable) -> Callable:
    @functools.wraps(fonksiyon)
    def wrapper(*args, **kwargs):
        print(f"[LOG] {fonksiyon.__name__} çağrılıyor.")
        return fonksiyon(*args, **kwargs)
    return wrapper

@logla
def mod_bulma(a: int, b: int) -> int:
    return a % b

print(mod_bulma(6, 5))      # Argümanları verirken üstte '(...) -> Any' yazıyor.

help(mod_bulma)     # Çıktı:  ... mod_bulma(a: int, b: int) -> int
# functools.wraps yazmasaydık Callable etkisiz kalcaktı. Wrapper'ın kendi imzasını gösterecekti.


# 2) Spesifik Kullanım - İmzayı Belirmek 
def logla(fonksiyon: Callable[[int, int], int]) -> Callable[[int, int],int]:
    @functools.wraps(fonksiyon)
    def wrapper(*args, **kwargs):
        print(f"[LOG] {fonksiyon.__name__} çağrılıyor.")
        return fonksiyon(*args, **kwargs)
    return wrapper

@logla
def mod_bulma(a: int, b: int) -> int:
    return a % b

print(mod_bulma(6, 5))  # Argümanları verirken üstte '(int, int) -> int' yazıyor.

[LOG] mod_bulma çağrılıyor.
1
Help on function mod_bulma in module __main__:

mod_bulma(a: int, b: int) -> int

[LOG] mod_bulma çağrılıyor.
1


## Bölüm 4.2: Parametreli ve Çoklu Decoratorler
--- 
### *Argüman alan decorator (decorator factory)*

* Şimdiye kadar decoratorların hepsi parametresizdi. Örn: @logla
* Lakin `@logla(seviye="DEBUG")` gibi kendi parametrelerini alan bir decorator yazmak için decorator fabrikası kurmamız gerekiyor.
* Yapı üç katmanlıdır:
    1. En dış katman decorator'un kendi parametrelerini alır.
    2. Orta katman asıl decorator'dur.
    3. En iç katman da wrapper'dır ve sarılacak fonksiyonu alır, *args ve **kwargs ile çağrıyı yürütür.  

In [37]:
import functools 

def tekrar_et(kac_kez):                 # Katman 1: Decorator'un kendi parametresi
    def decorator(fonksiyon):           # Katman 2: Asıl decorator. Fonksiyonu alır. 
        @functools.wraps(fonksiyon) 
        def wrapper(*args, **kwargs):   # Katman 3: Gerçek çağrıyı yürütür.
            sonuclar = []
            for _ in range(kac_kez):
                sonuclar.append(fonksiyon(*args, **kwargs))
            return sonuclar
        return wrapper
    return decorator

@tekrar_et(3)
def zar_at():
    import random
    return random.randint(1, 6)

# Uzun hali: zar_at = tekrar_et(3)(zar_at)
print(zar_at())

@tekrar_et              # Doğrusu -> @tekrar_et(1)
def bozuk_fonksiyon():
    return 1

try:
    bozuk_fonksiyon()
except TypeError as hata_mesaji:
    print(f"Hata: {hata_mesaji}") 

[5, 4, 4]
Hata: tekrar_et.<locals>.decorator() missing 1 required positional argument: 'fonksiyon'


### *Decorator istifleme (stacking) ve uygulanma sırası*

* Bir fonksiyona birden fazla decorator üst üste yazabiliyoruz.
* Bu decoratorlerin hangi sırayla çalıştığını anlamak çok önemli.
> İstiflenmiş decoratorlar fonksiyon tanımına en yakın olandan başlayarak aşağıdan yukarıya doğru uygulanır ama çalıştırırken yukarıdan aşağıya işler.

```python
@A 
@B 
def fonksiyon():
```
bu şuna eşittir: A(B(fonksiyon))
* fonksiyon() çağrıldığında ilk önce A devreye girer sonra B devreye girer en son da orijinal fonksiyon()'umuzun kodları çalışır.


In [38]:
# Decoratorler hangi sırayla uygulanır / çalışır?
import functools 
import time

# İki farklı decorator tanımlıyoruz.
# 1. decorator: Parametresiz. Sardığı fonksiyonun süresini ölçer.
# 2. decorator: Parametreli('rol'): Fonksiyonun çalışması için yetki kontrolü yapar.

def zaman_olc(fonksiyon):
    @functools.wraps(fonksiyon)
    def wrapper(*args, **kwargs):
        baslangic = time.perf_counter()
        sonuc = fonksiyon(*args, *kwargs)
        gecen_sure = time.perf_counter() - baslangic
        print(f"[Süre] {fonksiyon.__name__}: {gecen_sure:.4f} sn")
        return sonuc
    return wrapper

def yetki_gerekli(rol):
    def decorator(fonksiyon):
        @functools.wraps(fonksiyon)
        def wrapper(kullanici, *args, **kwargs):
            if kullanici.get("rol") != rol:
                raise PermissionError(f"'{rol}' rolü gerekli, mevcut rol: '{kullanici.get("rol")}'")
            return fonksiyon(kullanici, *args, **kwargs)
        return wrapper
    return decorator

# --- Uygulama -> Dekoratörleri tek bir fonksiyon için nasıl istifleriz? ---
@yetki_gerekli("admin")             # Dış katman: Önce bu çalışır, yetki yoksa akışı keser. 
@zaman_olc                          # İç katman: Yetki varsa süreyi ölçmeye başlar.
def rapor_olustur(kullanici):
    time.sleep(0.05)
    return f"{kullanici["ad"]} için rapor hazırlandı."

admin = dict(ad="Ayşe", rol="admin")
misafir = dict(ad="Ali", rol="misafir")

print(rapor_olustur(admin))
try: 
    rapor_olustur(misafir)
except PermissionError as hata_mesaji:
    print(f"Hata: {hata_mesaji}")

[Süre] rapor_olustur: 0.0505 sn
Ayşe için rapor hazırlandı.
Hata: 'admin' rolü gerekli, mevcut rol: 'misafir'


### *Class-based decorator (\_\_call\_\_ ile)*

* Decoratorler closure değil sınıf olarak da yazılabilir. Özellikle şu durumlarda:
    - Decorator kendi başına karmaşık, kalıcı bir state taşıması gerekiyorsa... <br>
    <u>Nasıl bir state?</u>
    - çağrı sayısı
    - configuration
    - diğer iç istatistikler 

> Bir sınıfı decorator olarak kullanmak için:
> 1. `__init__` sarılacak fonksiyonu ya da decorator parametrelerini alır.
> 2. `__call__` ise wrapper'in yaptığı işi üstlenir, gelen *args ve **kwargs'ji orijinal fonksiyona iletir.
> * Fonksiyon tabanlı decorator'da state bir cell'de(closure) saklanırken, sınıf tabanlı decorator'de state öznitelik olarak self.x şeklinde saklanır.

In [39]:
import functools 

class CagriSayaci:      # Sınıflar burada da PascalCase 
    
    def __init__(self, fonksiyon):
        functools.update_wrapper(self, fonksiyon)   #functool.wraps'in sınıflar için karşılığı
        self.fonksiyon = fonksiyon
        self.cagri_sayisi = 0
        # Buraya sarmalanmış her fonksiyon için tutulacak başka state'ler de tanımlayabiliriz.

    def __call__(self, *args, **kwargs):
        self.cagri_sayisi += 1
        print(f"[Çağrı #{self.cagri_sayisi}] {self.fonksiyon.__name__}")
        return self.fonksiyon(*args, **kwargs) 

@CagriSayaci
def merhaba_de(ad):
    return f"Merhaba, {ad}"       

print(merhaba_de("Ali"))
print(merhaba_de("Ayşe"))

@CagriSayaci
def gorusuruz_de(ad):
    return f"Görüşmek üzere, {ad}"
print(gorusuruz_de("Ali"))

print(merhaba_de.cagri_sayisi)      
# Direkt CagriSayaci sınıfının bir nesnesi olarak, merhaba_de'nin özniteliğine erişebiliyoruz.

[Çağrı #1] merhaba_de
Merhaba, Ali
[Çağrı #2] merhaba_de
Merhaba, Ayşe
[Çağrı #1] gorusuruz_de
Görüşmek üzere, Ali
2


### *Metotlarda decorator kullanımı ve `self` ile etkileşim* - Ek Konu

* Bir decorator'u bir sınıfa özgü fonksiyona (sınıfa ait metoda) da kullanabiliriz.
* Burada şuna dikkat edilmeli:
    - Python bir metodu nesne.metod() şeklinde çağırdığında nesne'nin kendisini otomatik olarak ilk argüman olarak dahil eder. 
    - Bu argümanı karşılayacak parametre olarak wrapper'ın *args vardır, bu yüzden hiçbir sorun çıkmaz. Bu sayede decoratorler metodlar üzerinde de sorunsuz çalışır.

> 🎯 CORE: Yerleşik @staticmethod , @classmethod , @property — hepsi Python'ın kendi sunduğu, metodlara özel tasarlanmış decorator'lardır ve bu self -ilişkili incelikleri kütüphane düzeyinde doğru yönetirler; kendi genel amaçlı decorator'larını yazarken bu üçünü "yeniden icat etmeye" çalışmak yerine gerektiğinde doğrudan kullanmayı tercih et.     
<p>

> Sınıf tabanlı decoratorlar yine başka sınıfların metodlarını sarmak için kullanıldıklarında, sardıkları sınıfın o metodunun kendisinin yerini alır, decorator'ün self'inin hiç görünmemesine sebep olur ve beklenmedik davranışlara yol açabilir. Aşağıdaki kodda bu durumu denedim kendim.
> - Çözüm: Metodları sarmalarken closure tabanlı decoratorler kullan; sade/bağımsız fonksiyonlar için yine nesne tabanlı decoratorler kullanabilirsin.

In [40]:
import functools 

def logla(fonksiyon):
    @functools.wraps(fonksiyon)
    def wrapper(*args, **kwargs):
        print(f"[Log:] {fonksiyon.__name__} çağrılıyor, args = {args}")
        return fonksiyon(*args, **kwargs)
    return wrapper

class Hesap:
    def __init__(self, bakiye):
        self.bakiye = bakiye

    @logla
    def yatir(self, miktar):
        self.bakiye += miktar
        return self.bakiye

hesap1 = Hesap(100)
print(hesap1.yatir(500))    # 'self' de yollanıyor.


# --- Bir metoda bu sefer sınıf tabanlı bir decorator sarmayı deneyelim (Çalışmayacak) ---
class CagriSayaciMetod:
    def __init__(self, fonksiyon):
        functools.update_wrapper(self, fonksiyon)
        self.fonksiyon = fonksiyon
        self.sayi = 0
    def __call__(self, *args, **kwargs):
        self.sayi += 1
        return self.fonksiyon(*args, **kwargs)

class Hesap2:
    def __init__(self, bakiye):
        self.bakiye = bakiye

    @CagriSayaciMetod
    def yatir(self, miktar):
        self.bakiye += miktar
        return self.bakiye

hesap2 = Hesap2(13500)
# print(hesap2.yatir(650))
# Bir üstteki satır çalışmaz çünkü hesap2.yatir bir 'bound method' değil, bir CagriSataciMetod nesnesidir. 
# Tüm Hesap2 nesneleri aynı ÇağrıSayaciMethod nesnesini paylaştıklarından (Çünkü method üstünde @decorator
# olduğunda sadece bir defalığına sarılıp orijinal methodun yerine atanıyor.) tüm Hesap2 nesneleri için
# CagriSayaciMetod'a ait işlem sayıları aynı olurdu. Tüm Hesap2 müşterileri aynı işlem sayısını paylaşırlardı.
        

[Log:] yatir çağrılıyor, args = (<__main__.Hesap object at 0x00000182DA48ABA0>, 500)
600
